In [0]:
df=spark.read.table("workspace.sales.bronze_superstore")
display(df)

In [0]:
from pyspark.sql.functions import col, regexp_replace, to_date

df_clean = df.withColumn(
    "Order_Date_clean",
    to_date(
        regexp_replace(col("Order_Date"), "-", "/"),
        "M/d/yyyy"
    )
).withColumn(
    "Ship_Date_clean",
    to_date(
        regexp_replace(col("Ship_Date"), "-", "/"),
        "M/d/yyyy"
    )
)
display(df_clean)

In [0]:
df_clean.filter(col("Order_Date_clean").isNull()) \
    .select("Order_Date") \
    .display()

In [0]:
silver_df = df_clean.dropna(subset=["Order_ID"]).fillna({"Profit": 0, "Sales": 0, "Discount": 0}).dropDuplicates()
display(silver_df)

In [0]:
spark.sql("drop TABLE IF EXISTS workspace.sales.silver_superstore")

silver_df.write.format("delta").mode("overwrite") \
    .saveAsTable("workspace.silver.superstore")

display(spark.read.table("workspace.sales.silver_superstore"))